# 06 · TAT-LLM (pre-trained by its authors) + our executor, scored with the official scorer

**No training on our side.** TAT-LLM (Zhu et al., 2024) is LLaMA-2 already fine-tuned by its authors on FinQA, TAT-QA and TAT-DQA (`next-tat/tat-llm-7b-fft`, public). Its prompt is rebuilt exactly as in the authors' released files (identical for all 1,644 dev questions).

The authors report 72.6 EM on TAT-DQA, but their evaluation script accepts answers within ~1% (or off by x100) and ignores the predicted unit. Here every answer is graded by the **official, unit-aware scorer**, twice: with the model's own arithmetic, and with **our Python executor** running the model's equation.

Same 200 dev questions as notebooks 01 and 05. Rough time on a T4: model download ~5 min, answering ~25 min.

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
from finsight.infer import load_model, predict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def latest_adapter(run):
    """The finished adapter, or the newest checkpoint if training stopped midway."""
    import glob
    if os.path.exists(f"{run}/final"):
        return f"{run}/final"
    ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    assert ckpts, f"No adapter in {run} - run notebook 02 (training) first"
    print("NOTE: training has not finished; using", ckpts[-1])
    return ckpts[-1]
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
import json
from finsight import tatllm

MODEL_ID = "next-tat/tat-llm-7b-fft"    # or "next-tat/tat-llm-13b-fft" (slower, use batch_size=2)
N = 200
eval_qs = sample(q_dev, N)                # same sample as notebooks 01 and 05
name = MODEL_ID.split("/")[-1]
model, tok = tatllm.load(MODEL_ID)
raw = tatllm.predict(model, tok, docs_dev, eval_qs, f"{OUT}/{name}_dev{N}.jsonl", batch_size=4)

## Official scores: model's own arithmetic vs. our executor

In [ ]:
RESULTS = {}
base_file = f"{OUT}/zeroshot_Qwen2.5-7B-Instruct_dev{N}.jsonl"
if os.path.exists(base_file):
    base = {json.loads(l)["uid"]: json.loads(l) for l in open(base_file)}
    RESULTS["Qwen2.5-7B untrained, our format (notebook 01)"] = score(eval_qs, base)
own = tatllm.score_outputs(raw, use_executor=False)
exe = tatllm.score_outputs(raw, use_executor=True)
RESULTS[f"{name}, model's own arithmetic"] = score(eval_qs, own)
RESULTS[f"{name} + our Python executor"] = score(eval_qs, exe)
for label, r in RESULTS.items():
    print_report(r, label)
json.dump({k: {m: v for m, v in r.items() if m != "rows"} for k, r in RESULTS.items()},
          open(f"{OUT}/{name}_comparison_dev{N}.json", "w"), indent=1)

## Look at mistakes (with the executor)

In [ ]:
rows = {r["uid"]: r for r in RESULTS[f"{name} + our Python executor"]["rows"]}
wrong = [q for q in eval_qs if rows[q["uid"]]["em"] == 0]
print(len(wrong), "wrong of", len(eval_qs))
for q in wrong[:10]:
    p = exe[q["uid"]]
    print(f"[{q['answer_type']}] {q['question']}")
    print(f"   gold: {q['answer']} ({q['scale']}) | derivation {q['derivation']!r}")
    print(f"   pred: {p['answer']} ({p['scale']})")
    print("   raw:", p["raw"].replace(chr(10), " / ")[:400], "\n")